# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Access metadata attributes
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Croissant Schema Version: {meta.conforms_to}")
print(f"Published: {meta.date_published} | License: {meta.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

### List all record sets and sample fields

In [ ]:
# List all record sets and their fields using @id referencing as per Croissant schema
record_sets = list(dataset.metadata.record_sets())
print(f"Total record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set @id: {rs.id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else ''}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} | name: {field.name}")
    print('-' * 60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, we select the main protein table as the primary record set
# Find the protein record set by convention or description

# Map record set name to @id for exploration
record_set_id_to_name = {rs.id: getattr(rs, 'name', rs.id) for rs in record_sets}
print("Available Record Sets and their @id:")
for k, v in record_set_id_to_name.items():
    print(f"  {k}: {v}")

# Choose main record set (assuming first, as often there is a single major set in experiments)
main_record_set_id = record_sets[0].id
print(f"\nUsing main record set: {main_record_set_id}\n")

# Load all records for each record set to DataFrames
dataframes = {}

for rs in record_sets:
    rsid = rs.id
    try:
        records = list(dataset.records(record_set=rsid))  # Each record is a dict
        dataframes[rsid] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[rsid])} records for record set {rsid}")
    except Exception as e:
        print(f"Failed loading {rsid}: {e}")

# Show columns
print("\nMain DataFrame fields:")
print(dataframes[main_record_set_id].columns.tolist())

# Show data
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Typical numeric fields for a proteomics table might include `coverage`, `mw`, `length`, `counts`, or `abundance` fields. We will choose one to illustrate.

In [ ]:
from pandas.api.types import is_numeric_dtype
import numpy as np

# Find first numeric-like field in the main dataframe
df = dataframes[main_record_set_id]
numeric_candidate = None
for col in df.columns:
    if is_numeric_dtype(df[col]) or np.issubdtype(df[col].dtype, np.number):
        numeric_candidate = col
        break
# fallback: try columns that are likely numeric
if numeric_candidate is None:
    for col in df.columns:
        if any(x in col.lower() for x in ['coverage', 'mw', 'length', 'count', 'abund']):
            # Attempt conversion
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if df[col].notna().sum() > 0:
                    numeric_candidate = col
                    break
            except Exception as e:
                continue

if numeric_candidate is None:
    raise ValueError("Could not find a numeric field to analyze.")

numeric_field = numeric_candidate
print(f"Using numeric field for analysis: {numeric_field}")

threshold = df[numeric_field].quantile(0.75) if df[numeric_field].notna().sum() > 0 else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.3f} (75th percentile): {filtered_df.shape[0]} out of {df.shape[0]}")

filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()

print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field
group_field = None
for col in df.columns:
    if col != numeric_field and not is_numeric_dtype(df[col]) and df[col].nunique() < 20:
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped data by '{group_field}'; mean {numeric_field} per group:")
    print(grouped_df)
else:
    print("\nNo suitable group-by categorical field found with <20 unique values.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Violin/box plot by group if group field exists
if group_field:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df, palette='pastel')
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we:
- Loaded the Croissant dataset and examined its metadata (including record sets and fields by their `@id`s).
- Loaded the main protein record set into a DataFrame and inspected the field structure.
- Performed EDA by filtering and normalizing a key numeric field, and grouping by a relevant categorical field.
- Visualized distribution and group statistics, facilitating further domain-specific biomarker or protein abundance analyses.

For more extensive analyses, continue with statistical testing, cross-dataset linking, or custom subgroup selection as needed.